# =======
# Initializion
# =======

In [57]:
import pandas as pd
import numpy as np
import regex as re
import os

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

import meteostat as ms
from datetime import datetime, timedelta
import helper_functions as f

# spark biblioteker fra labene
from pyspark.sql import SparkSession, Row
from pyspark.sql.functions import (
    col, lit, when, count, sum as spark_sum, avg,
    min as spark_min, max as spark_max, round as spark_round,
    desc, asc, to_timestamp, to_date, hour, dayofweek,
    lower, upper, trim, concat, coalesce, explode, split,
    countDistinct, collect_list, first, input_file_name,
    regexp_extract
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, TimestampType, BooleanType, LongType
)

In [52]:
# Create the SparkSession - the entry point for all Spark functionality
spark = SparkSession.builder \
    .appName("DAT535-Project-FootballStats") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print(f"Spark Version: {spark.version}")
print(f"Application ID: {spark.sparkContext.applicationId}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

Spark Version: 4.3.0
Application ID: local-1794319752498
Spark UI: http://10.255.255.254:4040


# ========
# Bronze layer
# ========

### OpenData Football data

In [53]:
LA_LIGA_MATCHES_PATH = "../open-data/data/matches/11"

matches_df = spark.read.json(LA_LIGA_MATCHES_PATH, multiLine=True).select(
    col("match_id"),
    col("match_date"),
    col("kick_off"),
    col("stadium.name")
)

In [62]:
EVENTS_PATH = "../open-data/data/events"

match_ids = matches_df.select(
    col("match_id")
).collect()

paths = []
for row in match_ids:
    paths.append(os.path.join(EVENTS_PATH, (str(row.match_id) + ".json")))

events_df = spark.read.json(paths, multiLine=True) \
                        .withColumn("match_id", 
                        regexp_extract(input_file_name(), r"([0-9]+).json", 1)) \
                        .select(
                            col("match_id"),
                            col("pass.height"),
                            col("pass.length"),
                            col("duration"),
                            col("pass.outcome")
                        )

events_df.write.mode("overwrite").parquet("bronze")

26/11/10 17:11:45 WARN MemoryManager: Total allocation exceeds 95.00% (2,040,109,440 bytes) of heap memory
Scaling row group sizes to 95.00% for 16 writers
26/11/10 17:11:46 WARN MemoryManager: Total allocation exceeds 95.00% (2,040,109,440 bytes) of heap memory
Scaling row group sizes to 95.00% for 16 writers


### Meteostat Weather data

In [ ]:
matches_df["stadium"].nunique()

In [ ]:
locations = matches_df[["stadium", "match_date", "kick_off"]]

for i, row in locations.iterrows():
    lat, long = f.get_coordinates(row["stadium"])

    # Creates a datetime from a string (e.g. datetime.striptime("2026-11-03 17:07:01.000", "%Y-%m-%d %H:%M:%S.%f"))
    match_start = datetime.strptime(f"{row["match_date"]} {row["kick_off"]}", "%Y-%m-%d %H:%M:%S.%f") \
        .replace(minute=0, second=0, microsecond=0)
    POINT = ms.Point(lat, long)
    START = match_start - timedelta(hours=3)
    END = match_start + timedelta(hours=3)

    stations = ms.stations.nearby(POINT, limit=4)

    time_series = ms.hourly(stations, START, END)
    weather = ms.interpolate(time_series, POINT).fetch()

# =======
# Silver layer
# =======

# =======
# Gold layer
# =======